In [1]:
import numpy as np
import random

# ==========================================
# STEP 1: INITIALIZATION (SETUP DATA & MAP)
# ==========================================

# Seed for reproducibility
random.seed(42)
np.random.seed(42)

# Problem Configuration
NUM_BINS = 10         # Bins are numbered 1 to 10. Node 0 is the Central Depot.
TRUCK_CAPACITY = 15   # Maximum capacity Q for each truck

# Generate random coordinates for the Depot (Node 0) and Bins (Nodes 1-10)
# (x, y) coordinates on a 100x100 grid
coordinates = {
    0: (50, 50),     # Depot is dead center
    1: (15, 20), 2: (80, 10), 3: (90, 85), 4: (10, 75), 5: (30, 60),
    6: (70, 30), 7: (25, 85), 8: (65, 75), 9: (85, 40), 10: (40, 15)
}

# Waste accumulated at each bin (Demands q_i). Depot (Node 0) has 0 demand.
bin_demands = {0: 0, 1: 4, 2: 3, 3: 5, 4: 2, 5: 6, 6: 3, 7: 4, 8: 2, 9: 5, 10: 3}

# Compute Physical Distance Matrix (d_ij) using Euclidean distance
num_nodes = NUM_BINS + 1
distance_matrix = np.zeros((num_nodes, num_nodes))
for i in range(num_nodes):
    for j in range(num_nodes):
        if i == j:
            distance_matrix[i][j] = float('inf')  # Can't drive to yourself
        else:
            coord_i = coordinates[i]
            coord_j = coordinates[j]
            distance_matrix[i][j] = np.sqrt((coord_i[0] - coord_j[0])**2 + (coord_i[1] - coord_j[1])**2)

# ACO Control Parameters
ALPHA = 1.0        # Weight for Pheromone History (History Importance)
BETA = 3.0         # Weight for Heuristic Closeness (Distance Importance)
RHO = 0.1          # Pheromone Evaporation Rate
NUM_ANTS = 15      # Number of scout drivers per iteration
MAX_ITER = 100     # Maximum execution loop cycles

# Initialize Pheromone Strengths Matrix (tau_ij) to an equal baseline value (0.1)
pheromone_matrix = np.full((num_nodes, num_nodes), 0.1)


def calculate_tour_distance(full_schedule):
    """Helper function to calculate total mileage of an ant's full plan."""
    total_dist = 0.0
    for route in full_schedule:
        for k in range(len(route) - 1):
            total_dist += distance_matrix[route[k]][route[k+1]]
    return total_dist


# Global Trackers for the "Champion Driver"
global_best_trucks = float('inf')
global_best_distance = float('inf')
global_best_schedule = []

print("Starting Waste Collection Optimization via ACO...\n")

# ==========================================
# STEP 2: START TOUR LOOP (ITERATION CYCLE)
# ==========================================
for iteration in range(1, MAX_ITER + 1):
    iteration_schedules = []
    iteration_distances = []
    iteration_truck_counts = []

    for ant in range(NUM_ANTS):
        # Master checklist of bins that need cleaning
        unvisited_bins = set(range(1, num_nodes))

        ant_full_schedule = []  # Stores all routes taken by this ant
        current_node = 0        # Start at the central Depot (Node 0)
        current_route = [0]     # Initialize the first truck's trip path
        remaining_capacity = TRUCK_CAPACITY  # Fresh truck capacity Q

        while unvisited_bins:
            # ==========================================
            # STEP 3: CHECK TRUCK CAPACITY (FILTERING)
            # ==========================================
            valid_candidates = [bin_idx for bin_idx in unvisited_bins if bin_demands[bin_idx] <= remaining_capacity]

            # ==========================================
            # STEP 5: EMPTY TRUCK AT DEPOT (IF NONE FIT)
            # ==========================================
            if not valid_candidates:
                current_route.append(0)              # Drive back to Depot
                ant_full_schedule.append(current_route) # Close this truck's route

                current_node = 0                     # Location reset to Depot
                current_route = [0]                  # Start a brand new truck path
                remaining_capacity = TRUCK_CAPACITY  # Reset capacity back to Q
                continue                             # Re-evaluate remaining bins from depot

            # ==========================================
            # STEP 4: SELECT NEXT BIN (ROULETTE WHEEL)
            # ==========================================
            probabilities = []
            for candidate in valid_candidates:
                # Math: Score = (Pheromone^ALPHA) * ((1/Distance)^BETA)
                pheromone = pheromone_matrix[current_node][candidate]
                closeness = 1.0 / distance_matrix[current_node][candidate]
                score = (pheromone ** ALPHA) * (closeness ** BETA)
                probabilities.append(score)

            # Turn raw scores into percentages summing up to 1.0
            sum_scores = sum(probabilities)
            probabilities = [score / sum_scores for score in probabilities]

            # Roulette-wheel choice selection
            next_node = np.random.choice(valid_candidates, p=probabilities)

            # ==========================================
            # STEP 6: UPDATE NODE STATES (LOAD & TRAVEL)
            # ==========================================
            current_route.append(next_node)
            remaining_capacity -= bin_demands[next_node]
            unvisited_bins.remove(next_node)
            current_node = next_node

        # Complete the final active truck's trip back to the depot
        current_route.append(0)
        ant_full_schedule.append(current_route)

        # Save this scout's completed daily plan
        iteration_schedules.append(ant_full_schedule)
        iteration_truck_counts.append(len(ant_full_schedule))
        iteration_distances.append(calculate_tour_distance(ant_full_schedule))

    # ==========================================
    # STEP 7: EVALUATE SOLUTIONS (FIND BEST ANT)
    # ==========================================
    # Primary sort by fewest trucks, secondary sort by shortest distance
    best_ant_idx = min(
        range(NUM_ANTS),
        key=lambda idx: (iteration_truck_counts[idx], iteration_distances[idx])
    )

    iter_best_trucks = iteration_truck_counts[best_ant_idx]
    iter_best_dist = iteration_distances[best_ant_idx]
    iter_best_sched = iteration_schedules[best_ant_idx]

    # Check if this iteration's winner beats the global all-time record
    if (iter_best_trucks < global_best_trucks) or (iter_best_trucks == global_best_trucks and iter_best_dist < global_best_distance):
        global_best_trucks = iter_best_trucks
        global_best_distance = iter_best_dist
        global_best_schedule = iter_best_sched

    # ==========================================
    # STEP 8: EVAPORATE PHEROMONES
    # ==========================================
    pheromone_matrix *= (1.0 - RHO)

    # ==========================================
    # STEP 9: REINFORCE BEST PATHS (DEPOSIT PHEROMONE)
    # ==========================================
    # We leave a heavy scent loop exclusively along the current Iteration Best paths
    # Deposit amount formula = 100 / total_distance
    deposit_amount = 100.0 / iter_best_dist
    for route in iter_best_sched:
        for step in range(len(route) - 1):
            from_node = route[step]
            to_node = route[step+1]
            pheromone_matrix[from_node][to_node] += deposit_amount
            pheromone_matrix[to_node][from_node] += deposit_amount  # Bidirectional road network

    # Print intermittent progress logs
    if iteration % 20 == 0 or iteration == 1:
        print(f"Iteration {iteration:03d} | Best Fleet Size: {iter_best_trucks} Trucks | Min Distance: {iter_best_dist:.2f} km")

# ==========================================
# STEP 10: TERMINATION CHECK & FINAL OUTPUT
# ==========================================
print("\n" + "="*50)
print("OPTIMIZATION COMPLETE — FINAL RESULTS")
print("="*50)
print(f"Optimal Trucks Required: {global_best_trucks} Trucks")
print(f"Total Fleet Distance:   {global_best_distance:.2f} km")
print("\nScheduled Truck Routing Itinerary:")

for idx, route in enumerate(global_best_schedule, 1):
    # Calculate specific truck capacity usage
    load_carried = sum(bin_demands[node] for node in route)
    route_str = " -> ".join(str(node) for node in route)
    print(f"  🚚 Truck #{idx}: {route_str} (Load: {load_carried}/{TRUCK_CAPACITY})")
print("="*50)


Starting Waste Collection Optimization via ACO...

Iteration 001 | Best Fleet Size: 3 Trucks | Min Distance: 474.15 km
Iteration 020 | Best Fleet Size: 3 Trucks | Min Distance: 433.15 km
Iteration 040 | Best Fleet Size: 3 Trucks | Min Distance: 433.15 km
Iteration 060 | Best Fleet Size: 3 Trucks | Min Distance: 433.15 km
Iteration 080 | Best Fleet Size: 3 Trucks | Min Distance: 433.15 km
Iteration 100 | Best Fleet Size: 3 Trucks | Min Distance: 433.15 km

OPTIMIZATION COMPLETE — FINAL RESULTS
Optimal Trucks Required: 3 Trucks
Total Fleet Distance:   433.15 km

Scheduled Truck Routing Itinerary:
  🚚 Truck #1: 0 -> 5 -> 4 -> 7 -> 8 -> 0 (Load: 14/15)
  🚚 Truck #2: 0 -> 1 -> 10 -> 2 -> 6 -> 0 (Load: 13/15)
  🚚 Truck #3: 0 -> 9 -> 3 -> 0 (Load: 10/15)
